# 🛠️ Notebook 2: Amazon Shopping — Implementation

## 🛠️ Setup

```bash
cd 07-object-oriented-design/amazon-shopping
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


In [ ]:
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from enum import Enum
import itertools

class OrderStatus(Enum):
    PENDING=1; PAID=2; SHIPPED=3; CANCELLED=4

@dataclass
class Product:
    sku: str; name: str; price: float; stock: int

class Cart:
    def __init__(self):
        self.lines = {}  # sku -> qty
    def add(self, product: Product, qty: int):
        if qty > product.stock:
            raise RuntimeError('not enough stock')
        self.lines[product.sku] = self.lines.get(product.sku, 0) + qty
    def remove(self, sku):
        self.lines.pop(sku, None)
    def total(self, catalog):
        return sum(catalog[sku].price * q for sku, q in self.lines.items())

@dataclass
class Order:
    id: int
    lines: dict        # sku -> qty  (frozen at checkout)
    total: float       # frozen
    status: OrderStatus = OrderStatus.PENDING

# --- Strategy pattern for payment ---
class PaymentMethod(ABC):
    @abstractmethod
    def pay(self, amount): ...

class CreditCard(PaymentMethod):
    def __init__(self, number, cvv): self.number, self.cvv = number, cvv
    def pay(self, amount):
        print(f'  💳 charged ${amount} to ****{self.number[-4:]}')
        return True

class PayPal(PaymentMethod):
    def __init__(self, email): self.email = email
    def pay(self, amount):
        print(f'  🅿️  charged ${amount} to {self.email}')
        return True

class Store:
    _oid = itertools.count(1)
    def __init__(self, products):
        self.catalog = {p.sku: p for p in products}
        self.orders = []
    def checkout(self, cart: Cart, method: PaymentMethod) -> Order:
        # Reserve stock
        for sku, qty in cart.lines.items():
            if self.catalog[sku].stock < qty:
                raise RuntimeError(f'{sku} out of stock')
        for sku, qty in cart.lines.items():
            self.catalog[sku].stock -= qty
        order = Order(next(Store._oid), dict(cart.lines), cart.total(self.catalog))
        if method.pay(order.total):
            order.status = OrderStatus.PAID
        self.orders.append(order)
        return order


## Walk-through

In [ ]:
store = Store([
    Product('BOOK-1','Clean Code', 25, stock=3),
    Product('MUG-1','Coffee Mug', 10, stock=5),
])

cart = Cart()
cart.add(store.catalog['BOOK-1'], 2)
cart.add(store.catalog['MUG-1'], 1)
print('cart total:', cart.total(store.catalog))

order = store.checkout(cart, CreditCard('4111111111111111','123'))
print('order', order.id, 'status', order.status)

# Next shopper can still buy the remaining mug but not 2 books anymore
print('book stock left:', store.catalog['BOOK-1'].stock)


### Why this shape is nice
- `Store.checkout` is the **transaction boundary**: stock check + deduct + order create in one place.
- Payment is a **strategy** (add ApplePay without touching the store).
- `Order` freezes the price at checkout — protects against price changes after.